In [ ]:
import pandas as pd
import timm
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, f1_score
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm

from internal.data_types import HistologyDataset
from internal.persistence_manager import PersistenceManager

data = PersistenceManager.load_dataset()
train_loader = data.train_loader
val_loader = data.val_loader

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
is_cuda_available = torch.cuda.is_available()
print(f'Using device: {device}')

In [ ]:
num_classes = 4

model = timm.create_model(
    'convnext_tiny',        # or 'tf_efficientnetv2_s_in21k'
    pretrained=True,
    num_classes=num_classes
)
model = model.to(device)

In [ ]:
class_counts = torch.tensor([445, 414, 397, 156], dtype=torch.float32)
class_weights = (class_counts.sum() / class_counts)  # inverse frequency
class_weights = class_weights / class_weights.mean() # normalize a bit
class_weights = class_weights.to(device)

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [ ]:
optimizer = AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=20
)

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for imgs, labels in tqdm(loader, desc="Train", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(imgs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        all_preds.append(preds.detach().cpu())
        all_targets.append(labels.detach().cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro')

    print(f"[] t_loss={epoch_loss:.4f} | F1(macro)={f1:.4f} | Acc={acc:.4f}")

    return epoch_loss, acc, f1

@torch.no_grad()
def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for imgs, labels in tqdm(loader, desc="Val", leave=False):
        imgs = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(imgs)
        loss = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)

        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())
        all_targets.append(labels.cpu())

    all_preds = torch.cat(all_preds).numpy()
    all_targets = torch.cat(all_targets).numpy()

    epoch_loss = running_loss / len(loader.dataset)
    acc = accuracy_score(all_targets, all_preds)
    f1 = f1_score(all_targets, all_preds, average='macro')

    return epoch_loss, acc, f1


In [ ]:
EPOCHS = 20
best_f1 = 0.0
best_state = None

for epoch in range(1, EPOCHS+1):
    print(f"\nEpoch {epoch}/{EPOCHS}")
    train_loss, train_acc, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, device
    )
    val_loss, val_acc, val_f1 = validate(
        model, val_loader, criterion, device
    )
    scheduler.step()

    print(
        f"Train  loss={train_loss:.4f} acc={train_acc:.4f} f1={train_f1:.4f} | "
        f"Val loss={val_loss:.4f} acc={val_acc:.4f} f1={val_f1:.4f}"
    )

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_state = model.state_dict().copy()
        torch.save(best_state, "best_convnext_tiny.pth")
        print(f"  🔥 New best F1: {best_f1:.4f} – model saved.")


In [ ]:
test_dataset = HistologyDataset(
    data.test_df,
    transforms=data.val_test_transforms,  # same as validation
    is_train=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,        # or 32 if fits
    shuffle=False,
    num_workers=4,
    pin_memory=True
)


In [ ]:
all_sample_indices = []
all_pred_labels = []

with torch.no_grad():
    for imgs, sample_indices in test_loader:
        imgs = imgs.to(device, non_blocking=True)

        logits = model(imgs)
        preds = logits.argmax(dim=1).cpu().numpy()  # [B]

        for si, p in zip(sample_indices, preds):
            all_sample_indices.append(si)
            all_pred_labels.append(data.idx2label[int(p)])


In [ ]:
# Ensure ".png" in the name
sample_index_with_ext = [f"{si}.png" if not si.endswith(".png") else si
                         for si in all_sample_indices]

submission_df = pd.DataFrame({
    "sample_index": sample_index_with_ext,
    "label": all_pred_labels
})

submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())